## Data Splitting Strategy: Development, Validation, and Holdout

To build a robust and production-aligned credit risk model, I split the dataset into three parts:

+ **Development Set (70%)**  
  Used for all feature engineering, variable transformation, and model training.  

+ **Validation Set (20%)**  
  Held out during model building and used only for final model evaluation and hyperparameter tuning.  

+ **Holdout Set (10%)**  
  Reserved for post-modeling use cases such as score distribution monitoring, population stability (PSI), and simulation of future drift.

All splits are **stratified on the loan_status variable to maintain target distribution across sets. A fixed random_state ensures reproducibility.

The three partitions are saved as separate CSV files for reuse across notebooks or modeling scripts.

In [43]:
from sklearn.model_selection import train_test_split
import pandas as pd



In [44]:
data_base = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/loan_data.csv')

In [45]:
data_remaining, data_holdout = train_test_split(
    data_base, test_size=0.10, random_state=20, stratify=data_base['loan_status'])

data, data_val = train_test_split(
    data_remaining, test_size=0.20, random_state=20, stratify=data_remaining['loan_status'])

print("Development:", data.shape)
print("Validation:", data_val.shape)
print("Holdout:", data_holdout.shape)

Development: (32400, 14)
Validation: (8100, 14)
Holdout: (4500, 14)


In [46]:
data.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/loan_data_70.csv')
data_val.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/loan_data_20.csv')
data_holdout.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/loan_data_10.csv')

# Univariate Variable Screening Dataset (Excluding IV)


This section sets up a filtered and transformed dataset for initial univariate analysis before formal Information Value (IV) evaluation.

### Purpose:
To prepare a clean working dataset (dataToAnalyze) where categorical variables are:
+ Encoded using Weight of Evidence (WoE)
+ Scaled for readability
+ Filtered to remove high-cardinality features
+ Checked for data type compatibility

### Key Steps:
+ Subset to relevant features using domain knowledge (lstvar)
+ Replace missing values with -1 for continuity
+ Remove variables with too many unique categories (≥100)
+ Compute WoE mappings and IV for categorical features
+ Convert WOE features to float64 for modeling
+ Scale WOE scores (multiplied by 100) for interpretation

### Output:
This version is saved as Internal_test_0_datasetForUnivariateVSExceptIV.csv and serves as a checkpoint to:
+ Perform diagnostic plotting or scorecard binning
+ Compare future transformations (e.g., smoothing, bin consolidation)
+ Ensure variables are modeling-ready before multivariate selection


In [47]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [48]:
def calculate_woe_iv(dataset, feature, target):
    lst = []
    for i in range(dataset[feature].nunique()):
        val = list(dataset[feature].unique())[i]
        lst.append({
            'Value': val,
            'All': dataset[dataset[feature] == val].count()[feature],
            'Good': dataset[(dataset[feature] == val) & (dataset[target] == 0)].count()[feature],
            'Bad': dataset[(dataset[feature] == val) & (dataset[target] == 1)].count()[feature]
        })

    dset = pd.DataFrame(lst)
    dset['Distr_Good'] = (dset['Good'] + 0.5) / dset['Good'].sum()
    dset['Distr_Bad'] = (dset['Bad'] + 0.5) / dset['Bad'].sum()
    dset['WoE'] = np.log(dset['Distr_Good']/ dset['Distr_Bad'])
    dset['IV'] = (dset['Distr_Good'] - dset['Distr_Bad']) * dset['WoE']
    iv = dset['IV'].sum()

    dset = dset.sort_values(by='WoE')

    return dset[['Value', 'All', 'Good', 'Bad', 'WoE']], iv

In [49]:
lstvar = ['person_education',
 'person_income',
 'person_emp_exp',
 'person_home_ownership',
 'loan_amnt',
 'loan_intent',
 'loan_int_rate',
 'loan_percent_income',
 'cb_person_cred_hist_length',
 'credit_score',
 'previous_loan_defaults_on_file',
 'loan_status']

In [50]:
combined = data[lstvar]

In [51]:
combined.head()

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,Associate,63788.0,17,RENT,5000.0,EDUCATION,11.01,0.08,12.0,686,No,0
11,Associate,13113.0,0,OWN,4500.0,HOMEIMPROVEMENT,8.63,0.34,2.0,651,No,1
9010,High School,59603.0,0,RENT,8000.0,PERSONAL,14.96,0.13,2.0,570,No,1
7030,Bachelor,54919.0,0,RENT,6225.0,VENTURE,11.54,0.11,4.0,706,Yes,0
21143,Associate,55000.0,7,MORTGAGE,6000.0,PERSONAL,9.32,0.11,9.0,643,Yes,0


In [52]:
dataFiltered = combined

In [53]:
dataToAnalyze = dataFiltered

In [54]:
dataToAnalyze

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,Associate,63788.0,17,RENT,5000.0,EDUCATION,11.01,0.08,12.0,686,No,0
11,Associate,13113.0,0,OWN,4500.0,HOMEIMPROVEMENT,8.63,0.34,2.0,651,No,1
9010,High School,59603.0,0,RENT,8000.0,PERSONAL,14.96,0.13,2.0,570,No,1
7030,Bachelor,54919.0,0,RENT,6225.0,VENTURE,11.54,0.11,4.0,706,Yes,0
21143,Associate,55000.0,7,MORTGAGE,6000.0,PERSONAL,9.32,0.11,9.0,643,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2353,Bachelor,46214.0,3,RENT,2500.0,DEBTCONSOLIDATION,12.53,0.05,2.0,684,No,0
4958,High School,51367.0,4,RENT,5000.0,VENTURE,10.99,0.10,4.0,569,Yes,0
38539,Bachelor,88660.0,5,MORTGAGE,12000.0,EDUCATION,7.46,0.14,4.0,698,No,0
5937,Master,39754.0,0,RENT,5600.0,DEBTCONSOLIDATION,7.90,0.14,3.0,606,Yes,0


# 🧹 Missing Value Handling

For simplicity in the initial analysis, missing values are replaced with `-1`. This placeholder ensures downstream transformations like Weight of Evidence (WOE) won't fail due to nulls.

In [55]:
dataToAnalyze = dataToAnalyze.fillna(-1)

In [56]:
dataToAnalyze.head()

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,Associate,63788.0,17,RENT,5000.0,EDUCATION,11.01,0.08,12.0,686,No,0
11,Associate,13113.0,0,OWN,4500.0,HOMEIMPROVEMENT,8.63,0.34,2.0,651,No,1
9010,High School,59603.0,0,RENT,8000.0,PERSONAL,14.96,0.13,2.0,570,No,1
7030,Bachelor,54919.0,0,RENT,6225.0,VENTURE,11.54,0.11,4.0,706,Yes,0
21143,Associate,55000.0,7,MORTGAGE,6000.0,PERSONAL,9.32,0.11,9.0,643,Yes,0


In [57]:
dataToAnalyze.select_dtypes(include=['object'])

,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
32229,Associate,RENT,EDUCATION,No
11,Associate,OWN,HOMEIMPROVEMENT,No
9010,High School,RENT,PERSONAL,No
7030,Bachelor,RENT,VENTURE,Yes
21143,Associate,MORTGAGE,PERSONAL,Yes
...,...,...,...,...
2353,Bachelor,RENT,DEBTCONSOLIDATION,No
4958,High School,RENT,VENTURE,Yes
38539,Bachelor,MORTGAGE,EDUCATION,No
5937,Master,RENT,DEBTCONSOLIDATION,Yes


In [58]:
dictobject = dict(dataToAnalyze.select_dtypes(include=['object']).nunique())

### Categorical Feature Screening

Identify and count categorical variables based on the number of unique values. Features with extremely high cardinality (≥100 categories) are flagged for exclusion. I will remove variables with excessive distinct categories that are unlikely to be useful in interpretable or stable models.

In [59]:
dictobject

{'person_education': np.int64(5),
 'person_home_ownership': np.int64(4),
 'loan_intent': np.int64(6),
 'previous_loan_defaults_on_file': np.int64(2)}

In [60]:
lstobject = [k for (k,v) in dictobject.items() if v < 50]

In [61]:
lstobject

['person_education',
 'person_home_ownership',
 'loan_intent',
 'previous_loan_defaults_on_file']

In [62]:
lstToremove = [k for (k,v) in dictobject.items() if v >= 100]

In [63]:
lstToremove

[]

In [64]:
dataToAnalyze.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32400 entries, 32229 to 9797
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_education                32400 non-null  object 
 1   person_income                   32400 non-null  float64
 2   person_emp_exp                  32400 non-null  int64  
 3   person_home_ownership           32400 non-null  object 
 4   loan_amnt                       32400 non-null  float64
 5   loan_intent                     32400 non-null  object 
 6   loan_int_rate                   32400 non-null  float64
 7   loan_percent_income             32400 non-null  float64
 8   cb_person_cred_hist_length      32400 non-null  float64
 9   credit_score                    32400 non-null  int64  
 10  previous_loan_defaults_on_file  32400 non-null  object 
 11  loan_status                     32400 non-null  int64  
dtypes: float64(5), int64(3), object(4)

In [65]:
dataToAnalyze = dataToAnalyze.drop(columns = lstToremove)

In [66]:
lstobject = list(dataToAnalyze.select_dtypes(include=['object']).columns)

## WOE Transformation and IV Calculation

For each retained categorical variable, compute its Weight of Evidence (WOE) encoding and Information Value (IV). These help assess the predictive signal of each variable relative to the target (loan_status) and also prepare them for downstream modeling.

In [67]:
IVs = []
woe_mappings = []
features = []

In [68]:
for i in lstobject:
    try:
        woe, iv = calculate_woe_iv(dataToAnalyze, i, 'loan_status')
        woe_map = dict(zip(woe['Value'], woe['WoE']))

        dataToAnalyze.loc[:, i] = dataToAnalyze[i].map(woe_map)

        # Only append if everything above succeeded
        features.append(i)
        IVs.append(iv)
        woe_mappings.append(woe_map)

    except Exception as e:
        print(f"[SKIPPED] {i} due to error: {e}")


In [69]:
results = pd.DataFrame({
    'features': features,
    'iv': IVs,
    'woe_mappings': woe_mappings
})


In [70]:
results

,features,iv,woe_mappings
0,person_education,0.000071,"{'Bachelor': -0.01125528573660267, 'Doctorate'..."
1,person_home_ownership,0.428330,"{'OTHER': -0.6337237600891447, 'RENT': -0.5180..."
2,loan_intent,0.120107,"{'DEBTCONSOLIDATION': -0.3918775683341684, 'ME..."
3,previous_loan_defaults_on_file,6.658928,"{'No': -1.0571171548200862, 'Yes': 9.148130565..."


In [71]:
dataToAnalyze.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32400 entries, 32229 to 9797
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_education                32400 non-null  object 
 1   person_income                   32400 non-null  float64
 2   person_emp_exp                  32400 non-null  int64  
 3   person_home_ownership           32400 non-null  object 
 4   loan_amnt                       32400 non-null  float64
 5   loan_intent                     32400 non-null  object 
 6   loan_int_rate                   32400 non-null  float64
 7   loan_percent_income             32400 non-null  float64
 8   cb_person_cred_hist_length      32400 non-null  float64
 9   credit_score                    32400 non-null  int64  
 10  previous_loan_defaults_on_file  32400 non-null  object 
 11  loan_status                     32400 non-null  int64  
dtypes: float64(5), int64(3), object(4)

## Convert WOE Features to Numeric Type

WOE-encoded variables are cast to float64 to ensure compatibility with modeling tools of this pipeline.

In [72]:
dataToAnalyze[lstobject] = dataToAnalyze[lstobject].astype('float64')

In [73]:
dataToAnalyze.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32400 entries, 32229 to 9797
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_education                32400 non-null  float64
 1   person_income                   32400 non-null  float64
 2   person_emp_exp                  32400 non-null  int64  
 3   person_home_ownership           32400 non-null  float64
 4   loan_amnt                       32400 non-null  float64
 5   loan_intent                     32400 non-null  float64
 6   loan_int_rate                   32400 non-null  float64
 7   loan_percent_income             32400 non-null  float64
 8   cb_person_cred_hist_length      32400 non-null  float64
 9   credit_score                    32400 non-null  int64  
 10  previous_loan_defaults_on_file  32400 non-null  float64
 11  loan_status                     32400 non-null  int64  
dtypes: float64(9), int64(3)
memory usa

In [74]:
dataToAnalyze[lstobject]

,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
32229,0.010023,-0.518036,0.331300,-1.057117
11,0.010023,1.310153,-0.254573,-1.057117
9010,0.003691,-0.518036,0.115455,-1.057117
7030,-0.011255,-0.518036,0.551317,9.148131
21143,0.010023,0.770118,0.115455,9.148131
...,...,...,...,...
2353,-0.011255,-0.518036,-0.391878,-1.057117
4958,0.003691,-0.518036,0.551317,9.148131
38539,-0.011255,0.770118,0.331300,-1.057117
5937,-0.002617,-0.518036,-0.391878,9.148131


In [75]:
dataToAnalyze[lstobject].nunique()

,0
person_education,5
person_home_ownership,4
loan_intent,6
previous_loan_defaults_on_file,2


In [76]:
dataToAnalyze[lstobject].isna().sum()

,0
person_education,0
person_home_ownership,0
loan_intent,0
previous_loan_defaults_on_file,0


## Scaling for Visualization

For exploratory plots or stability metrics, WOE values are scaled by 100. This does **not** affect model training, it's just for readability and downstream index generation.

In [77]:
dataToAnalyze[lstobject] = dataToAnalyze[lstobject].apply(lambda x: x*100)

In [78]:
dataToAnalyze[lstobject]

,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
32229,1.002263,-51.803563,33.130018,-105.711715
11,1.002263,131.015335,-25.457321,-105.711715
9010,0.369066,-51.803563,11.545480,-105.711715
7030,-1.125529,-51.803563,55.131699,914.813057
21143,1.002263,77.011839,11.545480,914.813057
...,...,...,...,...
2353,-1.125529,-51.803563,-39.187757,-105.711715
4958,0.369066,-51.803563,55.131699,914.813057
38539,-1.125529,77.011839,33.130018,-105.711715
5937,-0.261705,-51.803563,-39.187757,914.813057


## Export Prepared Dataset

The final transformed dataset and corresponding IV mappings are saved to CSV files for use in later steps of the modeling pipeline.

In [79]:
dataToAnalyze.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_test_0_datasetForUnivariateVSExceptIV.csv')

In [80]:
results.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_test_0_univariateVS_woe_mappings_iv.csv')

## Exporting Dataset for Multivariate Modeling Pipelines (Excluding IV)



This section mirrors the univariate preparation process, but the output is intended for **multivariate models** such as XGBoost or Random Forest.

While the feature cleaning and WoE transformation logic are the same, this version is tailored for models that:

+ Handle multicollinearity
+ Can tolerate or benefit from many variables
+ Are less sensitive to monotonicity or binning assumptions

No additional annotation is included here, as the logic is identical to the univariate preparation, only the modeling purpose differs.

In [81]:
def calculate_woe_iv(dataset, feature, target):
    lst = []
    for i in range(dataset[feature].nunique()):
        val = list(dataset[feature].unique())[i]
        lst.append({
            'Value': val,
            'All': dataset[dataset[feature] == val].count()[feature],
            'Good': dataset[(dataset[feature] == val) & (dataset[target] == 0)].count()[feature],
            'Bad': dataset[(dataset[feature] == val) & (dataset[target] == 1)].count()[feature]
        })

    dset = pd.DataFrame(lst)
    dset['Distr_Good'] = (dset['Good'] + 0.5) / dset['Good'].sum()
    dset['Distr_Bad'] = (dset['Bad'] + 0.5) / dset['Bad'].sum()
    dset['WoE'] = np.log(dset['Distr_Good']/ dset['Distr_Bad'])
    dset['IV'] = (dset['Distr_Good'] - dset['Distr_Bad']) * dset['WoE']
    iv = dset['IV'].sum()

    dset = dset.sort_values(by='WoE')

    return dset[['Value', 'All', 'Good', 'Bad', 'WoE']], iv

In [82]:
lstvar = ['person_education',
 'person_income',
 'person_emp_exp',
 'person_home_ownership',
 'loan_amnt',
 'loan_intent',
 'loan_int_rate',
 'loan_percent_income',
 'cb_person_cred_hist_length',
 'credit_score',
 'previous_loan_defaults_on_file',
 'loan_status']

In [83]:
combined = data[lstvar]

In [84]:
combined.head()

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,Associate,63788.0,17,RENT,5000.0,EDUCATION,11.01,0.08,12.0,686,No,0
11,Associate,13113.0,0,OWN,4500.0,HOMEIMPROVEMENT,8.63,0.34,2.0,651,No,1
9010,High School,59603.0,0,RENT,8000.0,PERSONAL,14.96,0.13,2.0,570,No,1
7030,Bachelor,54919.0,0,RENT,6225.0,VENTURE,11.54,0.11,4.0,706,Yes,0
21143,Associate,55000.0,7,MORTGAGE,6000.0,PERSONAL,9.32,0.11,9.0,643,Yes,0


In [85]:
dataFiltered = combined

In [86]:
dataToAnalyze = dataFiltered.iloc[:, :]

In [87]:
dataToAnalyze

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,Associate,63788.0,17,RENT,5000.0,EDUCATION,11.01,0.08,12.0,686,No,0
11,Associate,13113.0,0,OWN,4500.0,HOMEIMPROVEMENT,8.63,0.34,2.0,651,No,1
9010,High School,59603.0,0,RENT,8000.0,PERSONAL,14.96,0.13,2.0,570,No,1
7030,Bachelor,54919.0,0,RENT,6225.0,VENTURE,11.54,0.11,4.0,706,Yes,0
21143,Associate,55000.0,7,MORTGAGE,6000.0,PERSONAL,9.32,0.11,9.0,643,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2353,Bachelor,46214.0,3,RENT,2500.0,DEBTCONSOLIDATION,12.53,0.05,2.0,684,No,0
4958,High School,51367.0,4,RENT,5000.0,VENTURE,10.99,0.10,4.0,569,Yes,0
38539,Bachelor,88660.0,5,MORTGAGE,12000.0,EDUCATION,7.46,0.14,4.0,698,No,0
5937,Master,39754.0,0,RENT,5600.0,DEBTCONSOLIDATION,7.90,0.14,3.0,606,Yes,0


In [88]:
dataToAnalyze = dataToAnalyze.fillna(-1)

In [89]:
dataToAnalyze.head()

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,Associate,63788.0,17,RENT,5000.0,EDUCATION,11.01,0.08,12.0,686,No,0
11,Associate,13113.0,0,OWN,4500.0,HOMEIMPROVEMENT,8.63,0.34,2.0,651,No,1
9010,High School,59603.0,0,RENT,8000.0,PERSONAL,14.96,0.13,2.0,570,No,1
7030,Bachelor,54919.0,0,RENT,6225.0,VENTURE,11.54,0.11,4.0,706,Yes,0
21143,Associate,55000.0,7,MORTGAGE,6000.0,PERSONAL,9.32,0.11,9.0,643,Yes,0


In [90]:
dataToAnalyze.select_dtypes(include=['object'])

,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
32229,Associate,RENT,EDUCATION,No
11,Associate,OWN,HOMEIMPROVEMENT,No
9010,High School,RENT,PERSONAL,No
7030,Bachelor,RENT,VENTURE,Yes
21143,Associate,MORTGAGE,PERSONAL,Yes
...,...,...,...,...
2353,Bachelor,RENT,DEBTCONSOLIDATION,No
4958,High School,RENT,VENTURE,Yes
38539,Bachelor,MORTGAGE,EDUCATION,No
5937,Master,RENT,DEBTCONSOLIDATION,Yes


In [91]:
dictobject = dict(dataToAnalyze.select_dtypes(include=['object']).nunique())

In [92]:
dictobject

{'person_education': np.int64(5),
 'person_home_ownership': np.int64(4),
 'loan_intent': np.int64(6),
 'previous_loan_defaults_on_file': np.int64(2)}

In [93]:
lstobject = [k for (k,v) in dictobject.items() if v < 50]

In [94]:
lstobject

['person_education',
 'person_home_ownership',
 'loan_intent',
 'previous_loan_defaults_on_file']

In [95]:
lstToremove = [k for (k,v) in dictobject.items() if v >= 100]

In [96]:
lstToremove

[]

In [97]:
lstobject

['person_education',
 'person_home_ownership',
 'loan_intent',
 'previous_loan_defaults_on_file']

In [98]:
dataToAnalyze.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32400 entries, 32229 to 9797
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_education                32400 non-null  object 
 1   person_income                   32400 non-null  float64
 2   person_emp_exp                  32400 non-null  int64  
 3   person_home_ownership           32400 non-null  object 
 4   loan_amnt                       32400 non-null  float64
 5   loan_intent                     32400 non-null  object 
 6   loan_int_rate                   32400 non-null  float64
 7   loan_percent_income             32400 non-null  float64
 8   cb_person_cred_hist_length      32400 non-null  float64
 9   credit_score                    32400 non-null  int64  
 10  previous_loan_defaults_on_file  32400 non-null  object 
 11  loan_status                     32400 non-null  int64  
dtypes: float64(5), int64(3), object(4)

In [99]:
dataToAnalyze = dataToAnalyze.drop(columns = lstToremove)

In [100]:
lstobject = list(dataToAnalyze.select_dtypes(include=['object']).columns)

In [101]:
lstobject

['person_education',
 'person_home_ownership',
 'loan_intent',
 'previous_loan_defaults_on_file']

In [102]:
IVs = []
woe_mappings = []
features = []

In [103]:
for i in lstobject:
    try:
        woe, iv = calculate_woe_iv(dataToAnalyze, i, 'loan_status')
        woe_map = dict(zip(woe['Value'], woe['WoE']))

        dataToAnalyze.loc[:, i] = dataToAnalyze[i].map(woe_map)

        # Only append if everything above succeeded
        features.append(i)
        IVs.append(iv)
        woe_mappings.append(woe_map)

    except Exception as e:
        print(f"[SKIPPED] {i} due to error: {e}")


In [104]:
results = pd.DataFrame({
    'features': features,
    'iv': IVs,
    'woe_mappings': woe_mappings
})

In [105]:
results

,features,iv,woe_mappings
0,person_education,0.000071,"{'Bachelor': -0.01125528573660267, 'Doctorate'..."
1,person_home_ownership,0.428330,"{'OTHER': -0.6337237600891447, 'RENT': -0.5180..."
2,loan_intent,0.120107,"{'DEBTCONSOLIDATION': -0.3918775683341684, 'ME..."
3,previous_loan_defaults_on_file,6.658928,"{'No': -1.0571171548200862, 'Yes': 9.148130565..."


In [106]:
dataToAnalyze[lstobject] = dataToAnalyze[lstobject].astype('float64')

In [107]:
dataToAnalyze[lstobject]

,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
32229,0.010023,-0.518036,0.331300,-1.057117
11,0.010023,1.310153,-0.254573,-1.057117
9010,0.003691,-0.518036,0.115455,-1.057117
7030,-0.011255,-0.518036,0.551317,9.148131
21143,0.010023,0.770118,0.115455,9.148131
...,...,...,...,...
2353,-0.011255,-0.518036,-0.391878,-1.057117
4958,0.003691,-0.518036,0.551317,9.148131
38539,-0.011255,0.770118,0.331300,-1.057117
5937,-0.002617,-0.518036,-0.391878,9.148131


In [108]:
dataToAnalyze[lstobject].nunique()

,0
person_education,5
person_home_ownership,4
loan_intent,6
previous_loan_defaults_on_file,2


In [109]:
dataToAnalyze[lstobject].isna().sum()

,0
person_education,0
person_home_ownership,0
loan_intent,0
previous_loan_defaults_on_file,0


In [110]:
dataToAnalyze[lstobject] = dataToAnalyze[lstobject].apply(lambda x: x*100)

In [111]:
dataToAnalyze[lstobject]

,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
32229,1.002263,-51.803563,33.130018,-105.711715
11,1.002263,131.015335,-25.457321,-105.711715
9010,0.369066,-51.803563,11.545480,-105.711715
7030,-1.125529,-51.803563,55.131699,914.813057
21143,1.002263,77.011839,11.545480,914.813057
...,...,...,...,...
2353,-1.125529,-51.803563,-39.187757,-105.711715
4958,0.369066,-51.803563,55.131699,914.813057
38539,-1.125529,77.011839,33.130018,-105.711715
5937,-0.261705,-51.803563,-39.187757,914.813057


In [112]:
dataToAnalyze.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_test_0_datasetForMultivariateVSExceptIV.csv')

In [114]:
results.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_test_0_multivariateVS_woe_mappings_iv.csv')